In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
pd.option_context('display.max_rows', None)
pd.option_context('display.max_columns', None)

### Load dataset, pre-process, and filter shortlisted NMIs

In [ ]:
filename = Path.cwd().parent / "Datasets" / "Clariti Consumption" / "LMS_2013-01-01_2026-03-24_HALF_HOUR_au.pq"
df = pd.read_parquet(filename, engine='fastparquet')
df.columns = df.columns.str.replace(" consumption", "")

# Remove NMIs 6102507141 and VAAA003225
print("Original number of NMIs: " + str(len(df.columns)-1))
df.drop(columns=["6102507141","VAAA003225"], inplace=True)
print("New number of NMIs: "  + str(len(df.columns)-1))

df['date'] = pd.to_datetime(df['date'])
df.sort_values(by='date', ascending=True)
print(f"Duplicated period: {df[df['date'].duplicated()]['date']}")

NMI_list = df.columns.tolist()[1:]
data = {}
for nmi in NMI_list:
    first_date = min(df[df[nmi]!=0]['date'])
    last_date = max(df[df[nmi]!=0]['date'])
    filtered_col = df[((df['date']>=first_date) & (df['date']<=last_date))][[nmi]]
    
    zeroConsumption = len(filtered_col[filtered_col[nmi]==0])
    percentZeroConsumption = round(zeroConsumption*100 / len(filtered_col),2)
    
    data[nmi] = {
        'missing values':df[nmi].isna().sum(),
        'first date': first_date,
        'last date': last_date,
        '# of zero': zeroConsumption,
        '% of zero': percentZeroConsumption,
    }
    
    try:
        df.loc[df['date'] < first_date, nmi] = np.nan
    except:
        continue

summary_df = pd.DataFrame.from_dict(data, orient='index')

conditions = [(summary_df['% of zero'] < 1), (summary_df['% of zero'] < 10), (summary_df['% of zero'] < 50), (summary_df['% of zero'] < 100) ]
status = ['Active', 'Mostly Active', 'Intermittent', 'Mostly inactive']
summary_df['Status'] = np.select(conditions, status, default='Dead')
summary_df.loc[summary_df['first date']==summary_df['last date'],'Status'] = 'Dead'

NMI_shortlist = summary_df[((summary_df['Status']=='Active') | (summary_df['Status']=='Mostly Active'))].index.to_list()

In [ ]:
total_df = df.set_index('date').sum(axis=1, numeric_only=True).to_frame(name='Total')

In [ ]:
print(f'Number of shortlisted NMIs: {len(NMI_shortlist)}')

In [ ]:
filtered_df = df[NMI_shortlist + ['date']].set_index('date')

In [ ]:
folder = Path.cwd() / "EDA Results"
folder.mkdir(parents=True, exist_ok=True)

### Functions

### Zero-value pattern checks

In [ ]:
# Filter zero values within the active period
data = {}
for nmi in NMI_shortlist:
    data[nmi] = df[df[nmi]==0]['date'].reset_index(drop=True)
    
zero_df = pd.DataFrame.from_dict(data)

# Calculate gaps between zeroes
# Group consecutive zeroes
data = {}
run_count = 1

for nmi in NMI_shortlist:
    zero_df[nmi + ' gap'] = zero_df[nmi].diff()/ pd.Timedelta(minutes=30)
    
    data[nmi + ' range'] = []
    data[nmi + ' run count'] = []
    for index, row in zero_df.iterrows():
        if index == 0:
            first_date = zero_df.loc[index,nmi]
            last_date = first_date
        elif row[nmi + ' gap'] == 1:
            last_date = zero_df.loc[index,nmi]
            run_count += 1
        else:
            data[nmi + ' range'].append(str(first_date) + ' to ' + str(last_date))
            data[nmi + ' run count'].append(run_count)
            first_date = zero_df.loc[index,nmi]
            last_date = first_date
            run_count = 1
            if row[nmi + ' gap'] != row[nmi + ' gap']:
                break
        
zero_df = pd.DataFrame.from_dict(data,orient='index').transpose()

In [ ]:
# Get max and mode zero runs
# Group NMIs based on zero runs
data = {}
for nmi in NMI_shortlist:
    runs_count = zero_df[nmi + ' range'].count()
    max_run = zero_df[nmi + ' run count'].max()
    date_max_run = zero_df[zero_df[nmi + ' run count']==max_run][nmi + ' range'].tolist()

    try:
        max_run_2 = zero_df[zero_df[nmi + ' run count']!=max_run][nmi + ' run count'].max()
        date_max_run_2 = zero_df[zero_df[nmi + ' run count']==max_run_2][nmi + ' range'].tolist()
    except:
        max_run_2 = 0
        date_max_run_2 = None
        
    mode_run = zero_df[nmi + ' run count'].mode()[0]
    date_mode_run = zero_df[zero_df[nmi + ' run count']==mode_run][nmi + ' range'].tolist()
    
    if runs_count < 100:
        if max_run <= 24:
            group = 1
        elif max_run <= 96:
            group = 2
        else:
            group = 3
    else:
        if max_run <= 24:
            group = 4
        elif max_run <= 96:
            group = 5
        else:
            group = 6

    data[nmi] = {
        'group': group,
        'count of zero runs' : runs_count,
        'max zero runs' : max_run,
        'date of max zero runs' : date_max_run,
        '2nd max zero runs' : max_run_2,
        'date of 2nd max zero runs' : date_max_run_2[:10] if max_run_2 > 0 else None,
        'mode zero runs' : mode_run,
        'date of mode zero runs' : date_mode_run[:10]
    }

zero_df = pd.DataFrame.from_dict(data,orient='index')
zero_df.sort_values(by='group', ascending=False, inplace=True)
zero_df.to_csv(folder / 'EDA_zero-value pattern checks.csv')
zero_df.head(25)

### Outlier

In [ ]:
# Identify IQR-rule outliers for one series.
def iqr_outliers(values):
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (values < lower) | (values > upper), lower, upper

# Count consecutive half-hour outlier runs.
def max_run(dates):
    if dates.empty:
        return 0
    groups = dates.diff().div(pd.Timedelta(minutes=30)).ne(1).cumsum()
    return groups.value_counts().max()

In [ ]:
# Summarise distribution shape and IQR outliers.
rows = []

for nmi in NMI_shortlist:
    values = df[nmi]
    mask, lower, upper = iqr_outliers(values)
    outlier_dates = df.loc[mask, "date"]
    outlier_count = mask.sum()
    outlier_pct = outlier_count / len(values) * 100
    skew = values.skew()
    run = max_run(outlier_dates)
    month = outlier_dates.dt.to_period("M").value_counts().idxmax() if outlier_count else pd.NA

    rows.append([
        nmi,
        "right-skewed" if skew > 1 else "balanced",
        outlier_count,
        round(outlier_pct, 2),
        round(lower, 3),
        round(upper, 3),
        run,
        str(month) if outlier_count else pd.NA,
        "clustered" if run > 1 else "isolated",
        "substantial" if outlier_pct >= 5 else "moderate" if outlier_pct >= 1 else "few",
    ])

outlier_table = pd.DataFrame(rows, columns=[
    "NMI", "shape", "outlier_count", "outlier_pct", "iqr_lower", "iqr_upper",
    "max_outlier_run", "main_outlier_month", "outlier_pattern", "outlier_level"
])

outlier_table

In [ ]:
# Compact interpretation per NMI.
interpretation = outlier_table[["NMI", "shape", "outlier_level", "outlier_pattern", "main_outlier_month"]]
interpretation

In [ ]:
# Overall outlier profile across shortlisted NMIs.
outlier_table[["shape", "outlier_level", "outlier_pattern"]].value_counts().reset_index(name="nmi_count")

In [ ]:
# Generate plots for every shortlisted NMI.
def plot_histbox(df,col):
    values = df[col].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].hist(values, bins=50)
    axes[0].set_title(f"{col} histogram")
    axes[0].set_xlabel("consumption")
    axes[0].set_ylabel("count")
    axes[1].boxplot(values, vert=False)
    axes[1].set_title(f"{col} boxplot")
    axes[1].set_xlabel("consumption")
    plt.tight_layout()
    
    plt.savefig(folder / f"{col} Histogram and Boxplot.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

plot_histbox(total_df,'Total')
for nmi in NMI_shortlist:
    plot_histbox(df,nmi)

### Full time series

In [ ]:
def plot_fulltimeseries(df, col, title):
    plt.figure(figsize=(10, 4))
    plt.plot(df[col]) # Plot the column data against the date index
    
    plt.xlabel('30-minute interval')
    plt.ylabel('consumption')
    plt.title(f"{title} Full Time-Series")

    plt.savefig(folder / f"{title} Full Time-Series.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_fulltimeseries(total_df, 'Total', 'Total')
for nmi in filtered_df:
    plot_fulltimeseries(filtered_df, nmi, nmi)

### Deep dive

In [ ]:
daily_df = filtered_df[NMI_shortlist].resample("D").sum()
week_df = filtered_df[NMI_shortlist].groupby([filtered_df.index.dayofweek < 5, filtered_df.index.hour]).mean()

In [ ]:
def plot_deepdive(title, col, week_df, yearly_mean_df,year_ToD_df,year_month_df):
    colors = plt.cm.tab20(np.linspace(0, 1, year_ToD_df['year'].nunique()))
    fig = plt.figure(figsize=(15, 10))
    axs = fig.subplots(2,2)
    fig.suptitle(f'{title}')
    
    axs[0,0].plot(week_df[col].loc[True], label = 'weekday') # Plot the average consumption for each hour of the day for weekdays
    axs[0,0].plot(week_df[col].loc[False], label = 'weekend') # Plot the average consumption for each hour of the day for weekends
    axs[0,0].set_xlabel('hour')
    axs[0,0].set_ylabel('avg consumption')
    axs[0,0].set_title("Weekday vs Weekend")
    axs[0,0].legend()
    axs[0,0].set_xticks(np.arange(0, 24, 2)) # Set x-ticks to be every 2 hours of the day

    axs[0,1].bar(yearly_mean_df['date'],yearly_mean_df[col])
    axs[0,1].set_title('Yearly Mean Daily Consumption')
    axs[0,1].set_xlabel('year')
    axs[0,1].set_ylabel('avg daily consumption')

    for i, year in enumerate(sorted(year_ToD_df['year'].unique())):
        temp = year_ToD_df[year_ToD_df['year'] == year]
        axs[1,0].plot(range(len(temp)), temp[col], label=str(year), color=colors[i], lw=2)
        temp = year_month_df[year_month_df['year'] == year]
        axs[1,1].plot(range(len(temp)), temp[col], label=str(year), color=colors[i], lw=2)

    axs[1,0].set_title('Average Daily Load Shape by Year')
    axs[1,0].set_xlabel('30-minute interval')
    axs[1,0].set_ylabel('avg consumption')
    axs[1,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    axs[1,1].set_title('Monthly Consumption by Year')
    axs[1,1].set_xlabel('month')
    axs[1,1].set_ylabel('consumption')
    axs[1,1].set_xticks(list(range(12)), labels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    
    plt.tight_layout()
    plt.savefig(folder / f"{title} Deep Dive.png", dpi=300, bbox_inches='tight')
    plt.show()
    

total_daily_df = total_df['Total'].resample("D").sum()
total_week_df = total_df['Total'].groupby([total_df.index.dayofweek < 5, total_df.index.hour]).mean().to_frame(name='Total')
yearly_mean_df = total_daily_df.groupby(total_daily_df.index.year).mean().reset_index()
year_ToD_df = total_df['Total'].groupby([total_df.index.year, total_df.index.time]).mean().to_frame(name='Total').reset_index(names=['year','time'])
year_month_df = total_df['Total'].groupby([total_df.index.year, total_df.index.month]).sum().to_frame(name='Total').reset_index(names=['year','month'])
plot_deepdive('Total', 'Total', total_week_df, yearly_mean_df,year_ToD_df,year_month_df)

for nmi in NMI_shortlist:
    yearly_mean_df = daily_df[nmi].groupby(daily_df.index.year).mean().reset_index()
    year_ToD_df = filtered_df[nmi].groupby([filtered_df.index.year, filtered_df.index.time]).mean().to_frame(name=nmi).reset_index(names=['year','time'])
    year_month_df = filtered_df[nmi].groupby([filtered_df.index.year, filtered_df.index.month]).sum().to_frame(name=nmi).reset_index(names=['year','month'])
    
    plot_deepdive(nmi, nmi, week_df, yearly_mean_df,year_ToD_df,year_month_df)

### Rolling Mean and Standard Deviation

In [ ]:
# Plot 30-day rolling mean and rolling standard deviation for each NMI
def plot_rolling(rolling_mean, rolling_std, col):
    plt.figure(figsize=(10, 3))
    plt.plot(rolling_mean[col], label='30-day rolling mean')
    plt.plot(rolling_std[col], label='30-day rolling std')
    plt.xlabel('year')
    plt.ylabel('consumption')
    plt.title(f"{col} Rolling Mean and STD")
    plt.legend()

    plt.savefig(folder / f"{col} 30-Day Rolling Mean and STD.png", dpi=300, bbox_inches='tight')
    plt.show()

rolling_mean = total_daily_df.rolling(window=30, min_periods=1).mean().to_frame(name='Total')
rolling_std = total_daily_df.rolling(window=30, min_periods=1).std().to_frame(name='Total')
plot_rolling(rolling_mean, rolling_std, 'Total')
    
rolling_mean = daily_df.rolling(window=30, min_periods=1).mean()
rolling_std = daily_df.rolling(window=30, min_periods=1).std()
for nmi in daily_df:
    plot_rolling(rolling_mean, rolling_std, nmi)

### Lag Correlation

In [ ]:
# Add one exact lagged version of a variable.
def add_lag(data, variable, n):
    data[f"lag_{n}"] = data[variable].shift(n)

In [ ]:
# Compute current-vs-past correlations for each shortlisted NMI.
lags = [1, 2, 48, 336]
rows = []

for nmi in NMI_shortlist:
    data = df[[nmi]].copy()
    for n in lags:
        add_lag(data, nmi, n)
    rows.append([nmi, *data.corr().loc[nmi, [f"lag_{n}" for n in lags]]])

corr_table = pd.DataFrame(rows, columns=["NMI", *[f"lag_{n}" for n in lags]]).round(3)
corr_table

In [ ]:
# Label the strongest lag horizon.
meaning = {
    "1": "half-hour",
    "2": "one-hour",
    "48": "day",
    "336": "week",
}

def interpret(row):
    lag = row[[f"lag_{n}" for n in lags]].abs().idxmax().removeprefix("lag_")
    return meaning[lag]

interpretation = corr_table.assign(interpretation=corr_table.apply(interpret, axis=1))[["NMI", "interpretation"]]
interpretation

In [ ]:
# Count which lag is strongest across the shortlist.
corr_table[[f"lag_{n}" for n in lags]].abs().idxmax(axis=1).value_counts()